In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("bronze_data").getOrCreate()
df_bronze = spark.read.parquet("../../data/bronze/bronze.parquet")
df_bronze.show(5)
df_bronze.count()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/23 19:18:59 WARN Utils: Your hostname, ELBAHIA, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/01/23 19:18:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/23 19:19:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/23 19:19:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/01/23 19:19:09 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


+-------------------+--------+--------+--------+--------+-------+--------------------+------------------+----------------+---------------------+----------------------+------+
|          open_time|    open|    high|     low|   close| volume|          close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|
+-------------------+--------+--------+--------+--------+-------+--------------------+------------------+----------------+---------------------+----------------------+------+
|2026-01-23 08:11:00|89564.07|89590.03|89564.07|89590.02|4.57869|2026-01-23 08:11:...|    410104.5889242|             736|              4.09991|        367221.7281574|     0|
|2026-01-23 08:12:00|89590.02|89590.03|89550.13|89552.26|6.02776|2026-01-23 08:12:...|    539891.1861367|            2368|              0.83821|         75078.3275165|     0|
|2026-01-23 08:13:00|89552.27|89585.12|89542.27|89585.11|4.39672|2026-01-23 08:13:...|    393775.5325997|            1525|   

600

In [3]:
from pyspark.sql.functions import col

# Compter le nombre total de lignes
total_rows = df_bronze.count()
distinct_rows = df_bronze.dropDuplicates().count()

if total_rows != distinct_rows:
    print(f"Il y a {total_rows - distinct_rows} lignes dupliquées")
else:
    print("Aucune duplication détectée")


Aucune duplication détectée


In [4]:

# df_silver = df_bronze.orderBy("open_time").dropDuplicates(["open_time"])

In [5]:
# Filtrer uniquement les lignes qui ont au moins une valeur nulle
df_bronze.filter(
    sum([col(c).isNull().cast("int") for c in df_bronze.columns]) > 0
).show()
# df_bronze_clean = df_bronze.dropna()


+---------+----+----+---+-----+------+----------+------------------+----------------+---------------------+----------------------+------+
|open_time|open|high|low|close|volume|close_time|quote_asset_volume|number_of_trades|taker_buy_base_volume|taker_buy_quote_volume|ignore|
+---------+----+----+---+-----+------+----------+------------------+----------------+---------------------+----------------------+------+
+---------+----+----+---+-----+------+----------+------------------+----------------+---------------------+----------------------+------+



In [6]:
df_bronze.write.mode("overwrite").parquet("../../data/silver/btc_silver/")
print("Table btc_raw sauvegardée")

Table btc_raw sauvegardée
